# Lecture 04 — k Nearest Neighbors (kNN)

**Term:** Fall 2025  
**Week/Topic:** Lecture 04
**Instructor:** Dr. Bushaj

---

### What we'll cover
- Load and preview the **RidingMowers** dataset
- Visualize Owner vs. Nonowner in feature space
- Data partitioning (stratified) and scaling **without leakage**
- kNN classification with `Pipeline`
- Hyperparameter tuning for **k** (`GridSearchCV`)
- Evaluation: accuracy, confusion matrix, classification report
- Nearest neighbor lookup for a **new household**
- Decision boundary visualization

## Import required packages

In [ ]:
%matplotlib inline

import math
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier, KNeighborsRegressor
import matplotlib.pylab as plt

## Read Data

In [ ]:
my_drive_path = "YOUR_FILE_PATH_HERE"

In [ ]:
mower_df = pd.read_csv(my_drive_path+'RidingMowers.csv')
mower_df['Number'] = mower_df.index + 1
mower_df

In [ ]:
trainData, validData = train_test_split(mower_df, test_size=0.4, random_state=1)
print(trainData.shape, validData.shape)

In [ ]:
newHousehold = pd.DataFrame({'Income': [60], 'Lot_Size': [20]})
newHousehold

## Plotting the data

In [ ]:
fig, ax = plt.subplots()

subset = trainData.loc[trainData['Ownership']=='Owner']
ax.scatter(subset.Income, subset.Lot_Size, marker='o', label='Owner', color='C1')

subset = trainData.loc[trainData['Ownership']=='Nonowner']
ax.scatter(subset.Income, subset.Lot_Size, marker='D', label='Nonowner', color='C0')

ax.scatter(newHousehold.Income, newHousehold.Lot_Size, marker='*', label='New household', color='black', s=150)

plt.xlabel('Income')  # set x-axis label
plt.ylabel('Lot_Size')  # set y-axis label
for _, row in trainData.iterrows():
    ax.annotate(row.Number, (row.Income + 2, row.Lot_Size))

handles, labels = ax.get_legend_handles_labels()
ax.set_xlim(40, 115)
ax.legend(handles, labels, loc=4)

plt.show()

In [ ]:
def plotDataset(ax, data, showLabel=True, **kwargs):
    subset = data.loc[data['Ownership']=='Owner']
    ax.scatter(subset.Income, subset.Lot_Size, marker='o', label='Owner' if showLabel else None, color='C1', **kwargs)

    subset = data.loc[data['Ownership']=='Nonowner']
    ax.scatter(subset.Income, subset.Lot_Size, marker='D', label='Nonowner' if showLabel else None, color='C0', **kwargs)

    plt.xlabel('Income')  # set x-axis label
    plt.ylabel('Lot_Size')  # set y-axis label
    for _, row in data.iterrows():
        ax.annotate(row.Number, (row.Income + 2, row.Lot_Size))

fig, ax = plt.subplots()

plotDataset(ax, trainData)
plotDataset(ax, validData, showLabel=False, facecolors='none')

ax.scatter(newHousehold.Income, newHousehold.Lot_Size, marker='*', label='New household', color='black', s=150)

plt.xlabel('Income')  # set x-axis label
plt.ylabel('Lot_Size')  # set y-axis label

handles, labels = ax.get_legend_handles_labels()
ax.set_xlim(40, 115)
ax.legend(handles, labels, loc=4)

plt.show()

## Normalize data

In [ ]:
scaler = preprocessing.StandardScaler()
scaler.fit(trainData[['Income', 'Lot_Size']])

In [ ]:
mowerNorm = pd.concat([pd.DataFrame(scaler.transform(mower_df[['Income', 'Lot_Size']]),
                                    columns=['zIncome', 'zLot_Size']),
                       mower_df[['Ownership', 'Number']]], axis=1)

In [ ]:
mowerNorm.head(5)

In [ ]:
trainNorm = mowerNorm.iloc[trainData.index]
validNorm = mowerNorm.iloc[validData.index]

In [ ]:
newHouseholdNorm = pd.DataFrame(scaler.transform(newHousehold), columns=['zIncome', 'zLot_Size'])
newHouseholdNorm

In [ ]:
mower_df

In [ ]:
trainNorm

In [ ]:
validNorm

In [ ]:
knn = NearestNeighbors(n_neighbors=3)
knn.fit(trainNorm[['zIncome', 'zLot_Size']])

In [ ]:
distances, indices = knn.kneighbors(newHouseholdNorm)
print(trainNorm.iloc[indices[0], :])

In [ ]:
#Initialize a data frame with two columns: `k` and `accuracy`

train_X = trainNorm[['zIncome', 'zLot_Size']]
train_y = trainNorm['Ownership']
valid_X = validNorm[['zIncome', 'zLot_Size']]
valid_y = validNorm['Ownership']

# Train a classifier for different values of k
results = []
for k in range(1, 15):
    knn = KNeighborsClassifier(n_neighbors=k).fit(train_X, train_y)
    results.append({
        'k': k,
        'accuracy': accuracy_score(valid_y, knn.predict(valid_X.values))
    })

# Convert results to a pandas data frame
results = pd.DataFrame(results)
print(results)

In [ ]:
# Retrain with full dataset
mower_X = mowerNorm[['zIncome', 'zLot_Size']]
mower_y = mowerNorm['Ownership']
knn = KNeighborsClassifier(n_neighbors=5).fit(mower_X, mower_y)
distances, indices = knn.kneighbors(newHouseholdNorm)
print(knn.predict(newHouseholdNorm))
print('Distances',distances)
print('Indices', indices)
print(mowerNorm.iloc[indices[0], :])

# Example:  Personal Loan Acceptance

Universal Bank is a relatively young bank growing rapidly in terms of overall customer acquisition. The majority of these customers are liability customers (depositors) with varying sizes of relationship with the bank. The customer base of asset customers (borrowers) is quite small, and the bank is interested in expanding this base rapidly to bring in more loan business. In particular, it wants to explore ways of converting its liability customers to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for liability customers showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise smarter campaigns with better target marketing. The goal is to use
k-NN to predict whether a new customer will accept a loan offer. This will serve as the basis for the design of a new campaign.

The file _UniversalBank.csv_ contains data on 5000 customers. The data include customer demographic information (age, income, etc.), the customer’s relationship with the bank (mortgage, securities account, etc.), and the customer response to the last personal loan campaign (Personal Loan). Among these 5000 customers, only 480 (= 9.6%) accepted the personal loan that was offered to them in the earlier campaign.

Partition the data into training (60%) and validation (40%) sets.

__7.2.a__ Consider the following customer:

Age = 40, Experience = 10, Income = 84, Family = 2, CCAvg = 2, Education_1= 0, Education_2 = 1, Education_3 = 0, Mortgage = 0, Securities Account = 0, CD Account = 0, Online = 1, and Credit Card = 1. Perform a k-NN classification with all predictors except ID and ZIP code using k = 1. Remember to transform categorical predictors with more than two categories into dummy variables first. Specify the success class as 1 (loan acceptance), and use the default cutoff value of 0.5. How would this customer be classified?

__Answer:__

#### Data preparation
Load the data and remove unnecessary columns (ID, ZIP Code). Split the data into training (60%) and validation (40%) sets (use `random_state=1`).

In [ ]:
# Load the data
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')

# Drop ID and zip code columns
bank_df = bank_df.drop(columns=['ID', 'ZIP Code'])

# Make sure that the result is as expected
bank_df.head()

In [ ]:
# modify column names
bank_df.columns = [c.replace(' ', '_').replace('=', '_') for c in bank_df.columns]
list(bank_df.columns)

In [ ]:
# create dummy variables for categorical variable, we consider Education as categorical variable
bank_df['Education'] = bank_df['Education'].astype('category')
bank_df = pd.get_dummies(bank_df, prefix_sep='_', drop_first=False)
bank_df.head()

In [ ]:
# split dataset into training (60%) and validation (40%) sets
train_df, valid_df = train_test_split(bank_df, test_size=0.4, random_state=1)
print('Training set:', train_df.shape, 'Validation set:', valid_df.shape)

In [ ]:
# new customer
newCustomer = pd.DataFrame([{'Age': 40, 'Experience': 10, 'Income': 84, 'Family': 2, 'CCAvg': 2, 'Mortgage': 0,
                             'Securities_Account': 0, 'CD_Account': 0, 'Online': 1, 'CreditCard': 1, 'Education_1': 0,
                             'Education_2': 1, 'Education_3': 0}],
                            columns=['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Mortgage', 'Securities_Account',
                                   'CD_Account', 'Online', 'CreditCard', 'Education_1', 'Education_2', 'Education_3'])
newCustomer

In [ ]:
# normalize training and validation sets. The transformation is trained using the training set only.
# if you don't convert the integer columns to real numbers (float64),
# the StandardScaler will raise a DataConversionWarning. This is expected
outcome = 'Personal_Loan'
predictors = list(bank_df.columns)
predictors.remove(outcome)

scaler = preprocessing.StandardScaler()
scaler.fit(train_df[predictors])
scaler.transform(train_df[predictors])
# Transform the predictors of training, validation and newCustomer
train_X = scaler.transform(train_df[predictors])
train_y = train_df[outcome]
valid_X = scaler.transform(valid_df[predictors])
valid_y = valid_df[outcome]
newCustomerNorm = pd.DataFrame(scaler.transform(newCustomer),
                               columns=['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Mortgage', 'Securities_Account',
                                   'CD_Account', 'Online', 'CreditCard', 'Education_1', 'Education_2', 'Education_3'])
print(newCustomerNorm)

In [ ]:
# k-NN using k = 1
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(train_X, train_y)

In [ ]:
# predicted class
knn.predict(newCustomerNorm.values)

In [ ]:
# predicted probability
knn.predict_proba(newCustomerNorm.values)

New customer is predicted to not accept a loan offer.

__7.2.b__ What is a choice of k that balances between overfitting and ignoring the predictor information?

__Answer__

In [ ]:
# Train a classifier for different values of k
results = []
for k in range(1, 20, 2):
    knn = KNeighborsClassifier(n_neighbors=k).fit(train_X, train_y)
    results.append({
        'k': k,
        'accuracy': accuracy_score(valid_y, knn.predict(valid_X))
    })

# Convert results to a pandas data frame
results = pd.DataFrame(results)
results

In [ ]:
# plot accuracy vs. k
_ = results.plot.scatter(x='k', y='accuracy', xlim=[0, 20])

We choose the best k, which minimizes the misclassification rate in the validation set. Our best k is `k=5`

__7.2.c.__ Show the confusion matrix for the validation data that results from using the best k.

__Answer__

In [ ]:
# k-NN model for k = 5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_X, train_y)

knnPredOpt = knn.predict(valid_X)
print(confusion_matrix(valid_y, knnPredOpt))
print('Accuracy :', accuracy_score(valid_y, knnPredOpt))

__7.2.d.__ Consider the following customer: Age = 40, Experience = 10, Income = 84, Family = 2, CCAvg = 2, Education_1 = 0, Education_2 = 1, Education_3 = 0, Mortgage = 0, Securities Account = 0, CD Account = 0, Online = 1 and Credit Card = 1. Classify the customer using the best k.

__Answer__

In [ ]:
# predicted class
knn.predict(newCustomerNorm.values)

In [ ]:
# predicted probability
knn.predict_proba(newCustomerNorm.values)

New customer is predicted to not accept a loan offer.

__7.2.e.__ Repartition the data, this time into training, validation, and test sets (50% : 30% : 20%). Apply the k-NN method with the k chosen above. Compare the confusion matrix of the test set with that of the training and validation sets. Comment on the differences and their reason.

__Answer__

In [ ]:
# partition the data into training (50%), validation (30%) and test (20%) sets
train_df, temp_df = train_test_split(bank_df, test_size=0.5, random_state=1)
valid_df, test_df = train_test_split(temp_df, test_size=0.4, random_state=1)

print('Training : ', train_df.shape)
print('Validation : ', valid_df.shape)
print('Test : ', test_df.shape)

In [ ]:
# normalize training and validation sets. The transformation is trained using the training set only.
# if you don't convert the integer columns to real numbers (float64),
# the StandardScaler will raise a DataConversionWarning. This is expected
outcome = 'Personal_Loan'
predictors = list(bank_df.columns)
predictors.remove(outcome)

scaler = preprocessing.StandardScaler()
scaler.fit(train_df[predictors])

# Transform the predictors of training validation and newCustomer
train_X = scaler.transform(train_df[predictors])
train_y = train_df[outcome]
valid_X = scaler.transform(valid_df[predictors])
valid_y = valid_df[outcome]
test_X = scaler.transform(test_df[predictors])
test_y = test_df[outcome]
test_X = pd.DataFrame(test_X, columns=['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Mortgage', 'Securities_Account',
                                   'CD_Account', 'Online', 'CreditCard', 'Education_1', 'Education_2', 'Education_3'])
test_y = pd.DataFrame(test_y, columns=['Personal_Loan'])

In [ ]:
# k-NN model for best k = 5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_X, train_y)

In [ ]:
# confusion matrix of training set
knnPredOpt = knn.predict(train_X)
print(confusion_matrix(train_y, knnPredOpt))
print('Accuracy :', accuracy_score(train_y, knnPredOpt))

In [ ]:
# confusion matrix of validation set
knnPredOpt = knn.predict(valid_X)
print(confusion_matrix(valid_y, knnPredOpt))
print('Accuracy :', accuracy_score(valid_y, knnPredOpt))

In [ ]:
# confusion matrix of test set
knnPredOpt = knn.predict(test_X.values)
print(confusion_matrix(test_y, knnPredOpt))
print('Accuracy :', accuracy_score(test_y, knnPredOpt))

The error rate increases from the training set to the validation set, but decreases from the validation set to the test set.  The differences are small, but this decreased performance, at least in the test set, is unexpected but we can ignore it as difference is very small - both the training and validation sets are used in setting the optimal k so there can be overfitting.  The test set was **not** used to select the optimal k.

# Problem 7.3 Predicting Housing Median Prices

The file _BostonHousing.csv_ contains information on over 500 census tracts in Boston, where for each tract multiple variables
are recorded. The last column (CAT. MEDV) was derived from MEDV, such that it obtains the value 1 if MEDV > 30 and 0 otherwise. Consider the goal of predicting the median value (MEDV) of a tract, given the information in the first 12 columns.
Partition the data into training (60%) and validation (40%) sets.

__7.3.a.__ Perform a k-NN prediction with all 12 predictors (ignore the CAT. MEDV column), trying values of k from 1 to 5. Make sure to normalize the data. What is the best k? What does it mean?

__Answer__

#### Data preparation
Load the data and remove unnecessary columns (CAT. MEDV). Split the data into training (60%) and validation (40%) sets (use `random_state=1`).

In [ ]:
# Load the data
house_df = pd.read_csv(my_drive_path + 'BostonHousing.csv')

# Drop CAT.MEDV column
house_df = house_df.drop(columns=['CAT. MEDV'])

# Make sure that the result is as expected
house_df.head()

In [ ]:
# split dataset into training (60%) and validation (40%) sets
train_df, valid_df = train_test_split(house_df, test_size=0.4, random_state=1)
print('Training set:', train_df.shape, 'Validation set:', valid_df.shape)

In [ ]:
# normalize training and validation sets. The transformation is trained using the training set only.
# if you don't convert the integer columns to real numbers (float64), the StandardScaler will raise a DataConversionWarning.
# This is expected
outcome = 'MEDV'
predictors = list(house_df.columns)
predictors.remove(outcome)

scaler = preprocessing.StandardScaler()
scaler.fit(train_df[predictors])

# Transform the predictors of training validation and newCustomer
train_X = scaler.transform(train_df[predictors])
train_y = train_df[outcome]
valid_X = scaler.transform(valid_df[predictors])
valid_y = valid_df[outcome]

In [ ]:
# Train a regressor for different values of k
results = []
for k in range(1, 6):
    knn = KNeighborsRegressor(n_neighbors=k).fit(train_X, train_y)
    results.append({
        'k': k,
        'RMSE': math.sqrt(mean_squared_error(valid_y, knn.predict(valid_X)))
    })

# Convert results to a pandas data frame
results = pd.DataFrame(results)
results

Here best `k = 3`. This means that, for a given record, MEDV is predicted by averaging the MEDVs for the 3 closest records, proximity being measured by the distance between the vectors of predictor values.

__7.3.b.__ Predict the MEDV for a tract with the following information, using the best k:

CRIM = 0.2, ZN = 0, INDUS = 7, CHAS = 0, NOX = 0.538, RM = 6, AGE = 62, DIS = 4.7, RAD = 4, TAX = 307, PTRATIO = 21, LSTAT = 10.

__Answer__

In [ ]:
# new tract
newTract = pd.DataFrame([{'CRIM': 0.2, 'ZN': 0, 'INDUS': 7, 'CHAS': 0, 'NOX': 0.538, 'RM': 6, 'AGE': 62, 'DIS': 4.7, 'RAD': 4,
                          'TAX': 307, 'PTRATIO': 21, 'LSTAT': 10}],
                       columns=['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'LSTAT'])
newTract

In [ ]:
# normalize new record
newTractNorm = pd.DataFrame(scaler.transform(newTract),
                               columns=['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'LSTAT'])

newTractNorm

In [ ]:
# train knn model with k=3
knn = KNeighborsRegressor(n_neighbors=3)
knn.fit(train_X, train_y)


In [ ]:
# predict value of new tract
knn.predict(newTractNorm)

The predicted price of new tract is $18.77k.

__7.3.c.__ If we used the above k-NN algorithm to score the training data, what would be the error of the training set?

__Answer__

In the training set, the error is zero because the training cases are matched to themselves.

__7.3.d.__ Why is the validation data error overly optimistic compared to the error rate when applying this k-NN predictor to new data?

__Answer__

The validation error measures the error for the "best k" among multiple k's tried out for the validation data, so that particular k is optimized for the particular validation data set that was used in selecting it. It may not be as suitable for the new data.

__7.3.e.__ If the purpose is to predict MEDV for several thousands of new tracts, what would be the disadvantage of using k-NN prediction? List the operations that the algorithm goes through in order to produce each prediction.

__Answer__

KNN does not yield a uniform rule that can be applied to each new record to be predicted -- the whole "model building" process has to be repeated for each new record to be classified.

Specifically, the algorithm must calculate the distance from a new record to each of the training records, select the n-closest training records, determine the average target value for the n-closest training records, then score that target value to the new record, then repeat this process for each of the new records.